In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/pollution_us_2000_2016.csv")

print("RAW DATA OVERVIEW")
print("Initial Shape:", df.shape)
print("\nColumns:")
print(df.columns)
print("\nData Types:")
print(df.dtypes)

missing_before = df.isnull().sum()
print("\nMissing Values (Before Cleaning):")
print(missing_before)
print("Total Missing (Before):", missing_before.sum())



print("\nHANDLING MISSING VALUES")

numeric_cols = df.select_dtypes(include=[np.number]).columns
categorical_cols = df.select_dtypes(include=["object"]).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].mean())

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

missing_after = df.isnull().sum()
print("\nMissing Values (After Cleaning):")
print(missing_after)
print("Total Missing (After):", missing_after.sum())

print("\nMissing Values Reduced By:",
      missing_before.sum() - missing_after.sum())

print("\nDUPLICATE REMOVAL")

before_dup = df.shape[0]
duplicate_count = df.duplicated().sum()
print("Duplicate Rows Found:", duplicate_count)

df = df.drop_duplicates()
after_dup = df.shape[0]

print("Rows Before Removing Duplicates:", before_dup)
print("Rows After Removing Duplicates:", after_dup)
print("Total Duplicates Removed:", before_dup - after_dup)

print("\nOUTLIER DETECTION (IQR METHOD)")

before_outlier = df.shape[0]
print("Rows Before Outlier Removal:", before_outlier)

def remove_outliers_iqr(data, columns):
    for col in columns:
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        data = data[(data[col] >= lower_bound) & (data[col] <= upper_bound)]

    return data

df = remove_outliers_iqr(df, numeric_cols)

after_outlier = df.shape[0]

print("Rows After Outlier Removal:", after_outlier)
print("Outliers Removed:", before_outlier - after_outlier)

print("\nDATA TRANSFORMATION")

df['Date Local'] = pd.to_datetime(df['Date Local'])
df['Year'] = df['Date Local'].dt.year
df['Month'] = df['Date Local'].dt.month

print("New Columns Added: Year, Month")

yearly_avg = df.groupby("Year")[[
    "CO Mean",
    "NO2 Mean",
    "SO2 Mean",
    "O3 Mean"
]].mean().reset_index()

print("\nYearly Aggregated Data (Preview):")
print(yearly_avg.head())

df.to_csv("pollution_cleaned.csv", index=False)
yearly_avg.to_csv("pollution_yearly_avg.csv", index=False)

print("\nFINAL DATASET READY")
print("Final Shape:", df.shape)

RAW DATA OVERVIEW
Initial Shape: (1746661, 29)

Columns:
Index(['Unnamed: 0', 'State Code', 'County Code', 'Site Num', 'Address',
       'State', 'County', 'City', 'Date Local', 'NO2 Units', 'NO2 Mean',
       'NO2 1st Max Value', 'NO2 1st Max Hour', 'NO2 AQI', 'O3 Units',
       'O3 Mean', 'O3 1st Max Value', 'O3 1st Max Hour', 'O3 AQI', 'SO2 Units',
       'SO2 Mean', 'SO2 1st Max Value', 'SO2 1st Max Hour', 'SO2 AQI',
       'CO Units', 'CO Mean', 'CO 1st Max Value', 'CO 1st Max Hour', 'CO AQI'],
      dtype='object')

Data Types:
Unnamed: 0             int64
State Code             int64
County Code            int64
Site Num               int64
Address               object
State                 object
County                object
City                  object
Date Local            object
NO2 Units             object
NO2 Mean             float64
NO2 1st Max Value    float64
NO2 1st Max Hour       int64
NO2 AQI                int64
O3 Units              object
O3 Mean              floa